In [ ]:
# Kaggle: ativa Internet + GPU e executa esta única célula.
# Compila na primeira execução; reaproveita o binário e modelo durante a sessão.
import os, re, sys, time, secrets, shutil, subprocess, pathlib, urllib.request
ROOT=pathlib.Path('/kaggle/working'); REPO=ROOT/'llama.cpp'; BUILD=REPO/'build'; MODEL_DIR=ROOT/'qwen-model'
MODEL_FILE='Huihui-Qwen3.8-27B-abliterated-UD-Q4_K_XL.gguf'; MODEL=MODEL_DIR/MODEL_FILE; SERVER=BUILD/'bin'/'llama-server'
for name in ('llama-server','cloudflared'):
    subprocess.run(['pkill','-f',name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
def run(cmd,**kw): subprocess.run(cmd,check=True,**kw)
if not SERVER.exists():
    if not REPO.exists(): run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',str(REPO)])
    run(['cmake','-S',str(REPO),'-B',str(BUILD),'-DGGML_CUDA=ON','-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_BUILD_TYPE=Release','-DLLAMA_BUILD_SERVER=ON','-DLLAMA_BUILD_TESTS=OFF','-DLLAMA_BUILD_EXAMPLES=OFF'],stdout=subprocess.DEVNULL)
    run(['cmake','--build',str(BUILD),'--config','Release','-j','2','--target','llama-server'],stdout=subprocess.DEVNULL)
try: import huggingface_hub
except ImportError: run([sys.executable,'-m','pip','install','-q','huggingface_hub'])
from huggingface_hub import hf_hub_download
MODEL_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=str(MODEL) if MODEL.exists() else hf_hub_download(repo_id='huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF',filename=MODEL_FILE,local_dir=str(MODEL_DIR))
log=open(ROOT/'llama-server.log','w')
cmd=[str(SERVER),'-m',MODEL_PATH,'--host','127.0.0.1','--port','8000','--alias','qwen3.8-27b-abliterated','--ctx-size','4096','--n-gpu-layers','all','--split-mode','layer','--tensor-split','1,1','--jinja','--parallel','1']
server_proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
import requests
for _ in range(360):
    if server_proc.poll() is not None: raise RuntimeError('llama-server terminou; consulta /kaggle/working/llama-server.log')
    try:
        if requests.get('http://127.0.0.1:8000/health',timeout=2).ok: break
    except requests.RequestException: pass
    time.sleep(5)
else: raise TimeoutError('Qwen ainda a carregar; consulta /kaggle/working/llama-server.log')
cloudflared=shutil.which('cloudflared')
if not cloudflared:
    deb=ROOT/'cloudflared.deb'; urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',str(deb)); run(['dpkg','-i',str(deb)],stdout=subprocess.DEVNULL); cloudflared=shutil.which('cloudflared')
tunnel_log=open(ROOT/'cloudflared.log','w'); tunnel=subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT,start_new_session=True)
base_url=None
for _ in range(90):
    time.sleep(2); log_text=(ROOT/'cloudflared.log').read_text(errors='ignore'); match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log_text)
    if match: base_url=match.group(0)+'/v1'; break
if not base_url: raise RuntimeError('Cloudflare não abriu; consulta /kaggle/working/cloudflared.log')
print(base_url)
